# DOH 골프 3D 회전 — 쉬운 버전 (가입 없음)

**하는 법:** 위에서부터 각 회색칸 왼쪽 **▶** 를 순서대로 누르세요. 그게 전부예요.

**중요:** 먼저 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**.

(가입/로그인 필요 없음. 3D 관절만 뽑는 MMPose 모델을 씁니다 = SMPL 등록 불필요.)


### 1칸. 설치 (몇 분 걸림, 글자 주르륵 = 정상)


In [ ]:
!pip install -U openmim -q
!mim install mmengine -q
!mim install 'mmcv>=2.0.1' -q
!mim install 'mmdet>=3.1.0' -q
!mim install 'mmpose>=1.1.0' -q
print('설치 끝')


### 2칸. 스윙 영상 올리기
누르면 파일 선택창 → 골프 스윙 mp4 고르기 (짧게 자른 게 빠름).


In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print('올린 영상:', VIDEO)


### 3칸. 3D 분석 (제일 오래 걸림)
영상에서 프레임별 3D 관절을 뽑아 joints.pkl 로 저장.


In [ ]:
from mmpose.apis import MMPoseInferencer
import numpy as np, pickle

inferencer = MMPoseInferencer(pose3d='human3d', device='cuda')
gen = inferencer(VIDEO, pred_out_dir='mmpose_preds', return_vis=False)

frames = []
for r in gen:
    preds = r['predictions'][0]        # 이 프레임의 사람들
    if not preds:
        continue
    kp = np.array(preds[0]['keypoints'])   # (관절, 3)  3D
    frames.append(kp)

J = np.array(frames)
pickle.dump({'joints': J}, open('joints.pkl', 'wb'))
print('프레임', J.shape[0], '· 관절', J.shape[1] if J.ndim==3 else '?', '· shape', J.shape)


### 4칸. 회전 숫자 뽑기
먼저 관절 배열 확인(check) → 그 다음 회전량.
**P1/P4/P7 프레임 번호**는 영상 보고 대략 넣으면 됨 (어드레스/백스윙탑/임팩트).
모르면 일단 그대로 두고 그래프만 봐도 됨.


In [ ]:
!wget -q https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/claude/ai-video-analysis-engine-wlr06k/pose3d_poc/wham_golf_rotation.py -O rot.py
!python rot.py joints.pkl --skeleton h36m --check


In [ ]:
# P4(백스윙탑) 프레임 번호를 넣으면 그 지점 회전 + 그래프가 나와요
!python rot.py joints.pkl --skeleton h36m --p1 0 --p4 0 --p7 0 --png rot.png
from IPython.display import Image; import os
if os.path.exists('rot.png'):
    display(Image('rot.png'))


### 5칸. 형한테 보낼 것
- **백스윙탑 흉곽 회전 숫자** (위 출력)
- **rot.png 그래프** 캡처
이거 두 개만 캡처해서 보내주면 됩니다.

> 빨간 글자(에러)가 나오면 그 화면을 통째로 캡처해서 보내주세요. 고쳐서 다시 드립니다.
